# Courtship song — paper-ready figures

Per paired courtship bout this notebook answers four questions:

1. Which fly is the male? (the singer — Drosophila courtship song is male-produced)
2. What song types is he performing? (pulse / sine / waggle / quiet)
3. What are his walking patterns during singing? (forward speed, turn rate, walking vs stopped)
4. How does his center-of-mass height change across song types?

Data source: the post-rescue relinked h5 produced by `rescue_identity_relink_inplace.py`.
All reusable logic lives in `utils/song_analysis.py`, `utils/sex_id.py`, `utils/locomotion.py`.
The sandbox notebook `Courtship_Song_Analysis.ipynb` is untouched; this is a clean sibling.


In [ ]:
# --- Setup: imports + matplotlib style + output dir ---
from __future__ import annotations

import os, sys, pickle, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Project imports (notebook runs from the repo root OR from notebooks/)
REPO_ROOT = Path('.').resolve()
if not (REPO_ROOT / 'utils').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.courtship_loader import (
    load_courtship_h5, pair_bouts, get_fields, analyze_pair, analyze_all_pairs,
)
from utils.pair_validity import PairValidityConfig
from utils.song_analysis import SongAnalysisConfig
from utils.sex_id import SexIdConfig
from utils.locomotion import LocomotionConfig
from utils.locomotion import walking_fraction

# Paper-ready matplotlib defaults: Type-1 / sans-serif / clean spines
mpl.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica'],
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 9,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.spines.right': False,
    'axes.spines.top': False,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'standard',
    'savefig.transparent': False,
})
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
# Colors shared across figures
SONG_COLORS = {
    'pulse':  '#d62728',  # red
    'sine':   '#1f77b4',  # blue
    'waggle': '#9467bd',  # purple
    'quiet':  '#bdbdbd',  # grey
}
# After the reorder in analyze_pair, slot 0 = male, slot 1 = female.
FLY_COLORS = {'fly0': '#ff7f0e', 'fly1': '#2ca02c'}  # male, female

FIG_DIR = REPO_ROOT / 'figures' / 'courtship'
FIG_DIR.mkdir(parents=True, exist_ok=True)

H5_PATH = Path('/data2/users/eabe/datasets/Johnson_lab/courtship/04092026_bouts_04112026/'
               'v1/'
               'ik_output_combined_v1_courtship_both.h5')

CACHE_PATH = FIG_DIR / 'per_bout_cache.pkl'

print(f'repo   : {REPO_ROOT}')
print(f'h5     : {H5_PATH}')
print(f'figures: {FIG_DIR}')


In [ ]:
from utils.courtship_loader import load_and_merge_courtship_h5
# --- Load the relinked h5 and pair fly0/fly1 bouts ---
# Each bout key in this h5 holds ONE fly's kp_data / xpos_egocentric / qpos.
# The pairing is consecutive (fly0, fly1) and validated by info['source_flies'].

print('loading h5 (slow — full file)...')
# data, info, kp_names, bout_keys = load_courtship_h5(H5_PATH)

H5_SESSION0 = Path('/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/v1/ik_output_combined_v1_courtship_both_full_exemplar.h5')
# H5_SESSION0 = Path('/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/Temp/v1/ik_output_combined_v1_courtship_both.h5')
H5_MAIN = Path('/data2/users/eabe/datasets/Johnson_lab/courtship/'
               'Session1/Session1_bouts_04172026/v1/ik_output_combined_v1_courtship_both.h5')

data, info, kp_names, bout_keys = load_and_merge_courtship_h5([H5_SESSION0, H5_MAIN])


pairs = pair_bouts(bout_keys, info)
print(f'bouts: {len(bout_keys)}  pairs: {len(pairs)}  kp: {len(kp_names)}')
print(f'first 3 pairs: {pairs[:3]}')


In [ ]:
# --- Run the full per-pair analysis and cache to a pickle ---
# Set FORCE=True to invalidate the cache and re-run from scratch.
FORCE = True

song_cfg = SongAnalysisConfig()
song_cfg.pipeline = "both"
sex_cfg  = SexIdConfig()
loc_cfg  = LocomotionConfig()
pair_cfg = PairValidityConfig()

results = analyze_all_pairs(
    data, pairs, kp_names,
    cache_path=CACHE_PATH,
    force=FORCE,
    song_cfg=song_cfg,
    sex_cfg=sex_cfg,
    loc_cfg=loc_cfg,
    pair_cfg=pair_cfg,
)

print(f'ok: {len(results)} pair results loaded')


In [ ]:
# --- Build a per-bout results dataframe for figures + CSV export ---
def _frac(labels, valid):
    labels = np.asarray(labels); valid = np.asarray(valid, dtype=bool)
    n = int(valid.sum())
    if n == 0:
        return {k: float('nan') for k in ('pulse', 'sine', 'waggle', 'quiet')}
    sel = labels[valid]
    return {k: float(np.mean(sel == k)) for k in ('pulse', 'sine', 'waggle', 'quiet')}


def _legacy_male_labels(r):
    """Return the legacy-detector's dominant-wing frame labels for the male,
    or None if the pair wasn't run through the legacy pipeline."""
    male_song = r['song0']  # post-swap: song0 is always male
    dom = str(male_song.get('dominant_wing', 'L')).upper()
    side_key = 'L' if dom.startswith('L') else 'R'
    side = male_song.get('sides', {}).get(side_key, {})
    lab = side.get('frame_labels_legacy')
    return None if lab is None else np.asarray(lab)


rows = []
for r in results:
    sex = r['sex']
    fr  = _frac(r['male_labels'], r['male_valid'])
    # Legacy labels are surfaced by the new parallel-output detector; fall
    # back to NaN columns when they aren't present (e.g. legacy disabled).
    _lab_leg = _legacy_male_labels(r)
    fr_leg = (_frac(_lab_leg, r['male_valid']) if _lab_leg is not None
              else {k: float('nan') for k in ('pulse', 'sine', 'waggle', 'quiet')})
    bs  = r['by_song']
    row = {
        'pair_idx':                   r['pair_idx'],
        # Post-swap keys: key0 is always the male's original bout, key1 the female's.
        'key0':                       r['key0'],
        'key1':                       r['key1'],
        'T':                          r['T'],
        'male_id':                    sex['male_id'],
        'female_id':                  sex['female_id'],
        'criterion':                  sex['criterion'],
        'confidence':                 sex['confidence'],
        'disagree_bodylen':           sex['disagree'],
        # Semantic (post-swap) song fractions. slot0=male, slot1=female.
        'song_fraction_male':         r['song0']['summary']['song_fraction'],
        'song_fraction_female':       r['song1']['summary']['song_fraction'],
        'body_length_male':           sex['body_length_male'],
        'body_length_female':         sex['body_length_female'],
        # Raw tracker-slot record for traceability + Figure 1.
        'tracker_male_id':            r['tracker_male_id'],
        'tracker_key0':               r['tracker_key0'],
        'tracker_key1':               r['tracker_key1'],
        'tracker_song_fraction_fly0': r['tracker_song_fraction_fly0'],
        'tracker_song_fraction_fly1': r['tracker_song_fraction_fly1'],
        'frac_pulse':                 fr['pulse'],
        'frac_sine':                  fr['sine'],
        'frac_waggle':                fr['waggle'],
        'frac_quiet':                 fr['quiet'],
        # Legacy-detector fractions for side-by-side comparison.
        'frac_pulse_legacy':          fr_leg['pulse'],
        'frac_sine_legacy':           fr_leg['sine'],
        'frac_waggle_legacy':         fr_leg['waggle'],
        'frac_quiet_legacy':          fr_leg['quiet'],
        'walking_fraction':           walking_fraction(r['walking_state'][r['male_valid']]) if r['male_valid'].any() else float('nan'),
        'mean_com_z':                 float(np.nanmean(r['com_z'][r['male_valid']])) if r['male_valid'].any() else float('nan'),
        'mean_speed_bl':              float(np.nanmean(r['kin'].get('speed_bl', r['kin']['speed'])[r['male_valid']])) if r['male_valid'].any() else float('nan'),
    }
    # Song-conditioned aggregates
    for song in ('pulse', 'sine', 'waggle', 'quiet'):
        stats = bs.get(song, {})
        for m in ('speed_bl', 'forward_speed_bl', 'turn_rate', 'com_z'):
            row[f'{song}_{m}_mean'] = stats.get(m, {}).get('mean', float('nan'))
            row[f'{song}_{m}_n']    = stats.get(m, {}).get('n',    0)
    rows.append(row)

df = pd.DataFrame(rows)
print(df.shape)
df.head(25)


In [ ]:
%matplotlib inline
EXAMPLE_IDX = 13 # 73

# Optional sub-range (in frames, bout-relative) to zoom into a clip of
# the example bout. Inherited by Figure 3 below. None = full bout.
# When trimmed, plotted arrays are sliced so y-axes auto-scale to the
# in-range data only (extreme values outside the range are dropped).
# START_FRAME = (150/1000) * 800  # inclusive
# END_FRAME   = (1500/1000) * 800  # exclusive; None → render to end of bout

if EXAMPLE_IDX is None:
    example_idx = int((df['frac_pulse'] + df['frac_sine']).idxmax())
else:
    example_idx = int(EXAMPLE_IDX)
ex = results[example_idx]
print(f'example pair_idx={example_idx}  keys={ex["key0"]}/{ex["key1"]}  '
      f'T={ex["T"]} ({ex["T"]/song_cfg.fs:.3f}s)  male={ex["sex"]["male_id"]}')

fs = song_cfg.fs
male_song = ex['song0'] if ex['sex']['male_id'] == 'fly0' else ex['song1']

# ----- plotting helpers (local — not exported) ----------------------------
_SEG_COLORS = {
    'pulse':  '#E76F5133',
    'sine':   '#2A9D8F33',
    'waggle': '#E9C46A33',
    'quiet':  '#cccccc11',
}
_SEG_EDGE_COLORS = {
    'pulse':  '#E76F51',
    'sine':   '#2A9D8F',
    'waggle': '#E9C46A',
    'quiet':  '#999999',
}
_WING_COLORS = {
    'WingL_V12': '#7B2D8E', 'WingL_V13': '#A855F7',
    'WingR_V12': '#0369A1', 'WingR_V13': '#38BDF8',
}


def _add_segment_shading(ax, segments, fs, hatch=None, side_tag=None,
                         seen_labels=None, frame_range=None):
    if seen_labels is None:
        seen_labels = set()
    for seg in segments:
        stype = seg['type']
        if stype == 'quiet':
            continue
        # Clip to the visible frame range if provided; drop segments that
        # fall entirely outside so they don't pollute the legend.
        if frame_range is not None:
            lo, hi = frame_range
            if seg['end'] <= lo or seg['start'] >= hi:
                continue
            seg_start = max(seg['start'], lo)
            seg_end   = min(seg['end'],   hi)
        else:
            seg_start = seg['start']
            seg_end   = seg['end']
        s_ms = seg_start / fs * 1000.0
        e_ms = seg_end   / fs * 1000.0
        color = _SEG_COLORS.get(stype, '#cccccc33')
        label_key = f'{stype}_{side_tag}'
        if label_key in seen_labels:
            label = None
        else:
            label = f'{stype.capitalize()} {side_tag}' if side_tag else stype.capitalize()
            seen_labels.add(label_key)
        ax.axvspan(s_ms, e_ms, facecolor=color, zorder=0, label=label,
                   hatch=hatch, edgecolor=_SEG_EDGE_COLORS.get(stype),
                   linewidth=0)


def _add_segment_freq_annotations(ax, segments, window_features, fs,
                                  frame_range=None):
    wc = window_features.get('window_centers')
    # New detector exposes the per-window multitaper peak as 'window_freq'
    # and a 'window_is_sine' mask; legacy detector exposes 'peak_freq' for all
    # windows. Support both so this helper works on new and legacy features.
    pf = window_features.get('window_freq')
    is_sine = window_features.get('window_is_sine')
    if pf is None:
        pf = window_features.get('peak_freq')
    if wc is None or pf is None or len(wc) == 0:
        return
    wc = np.asarray(wc); pf = np.asarray(pf)
    is_sine = np.asarray(is_sine) if is_sine is not None else None
    for seg in segments:
        stype = seg['type']
        if stype == 'quiet':
            continue
        # With the new detector, window_freq is only meaningful for sine
        # windows (pulse-masked windows are excluded from the F-test), so
        # skip pulse/waggle segments when the new schema is in use.
        if is_sine is not None and stype != 'sine':
            continue
        s, e = seg['start'], seg['end']
        if frame_range is not None:
            lo, hi = frame_range
            if e <= lo or s >= hi:
                continue
            s = max(s, lo); e = min(e, hi)
        in_seg = (wc >= s) & (wc <= e)
        if is_sine is not None:
            in_seg = in_seg & is_sine
        if in_seg.sum() == 0:
            continue
        seg_freqs = pf[in_seg]
        median_f = float(np.nanmedian(seg_freqs))
        min_f = float(np.nanmin(seg_freqs))
        max_f = float(np.nanmax(seg_freqs))
        mid_ms = (s + e) / 2.0 / fs * 1000.0
        edge_color = _SEG_EDGE_COLORS.get(stype, '#999999')
        freq_str = f'{median_f:.0f}Hz' if min_f == max_f else f'{min_f:.0f}-{max_f:.0f}Hz'
        ax.annotate(
            freq_str,
            xy=(mid_ms, 0.02), xycoords=('data', 'axes fraction'),
            fontsize=7, color=edge_color, fontweight='bold',
            ha='center', va='bottom',
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec=edge_color,
                      alpha=0.85, lw=0.7),
        )


def _dom_side(result):
    # Pulse is bilateral but the new detector runs the multitaper sine
    # detector independently on L and R, so their 'segments' lists
    # differ. Shade using the dominant wing so the visible segments match
    # the label track / summary for that bout.
    dom = str(result.get('dominant_wing', 'L')).upper()[0]
    if dom not in result.get('sides', {}):
        dom = 'L'
    return dom


def _shade_song(ax, result, fs, seen, frame_range=None):
    dom = _dom_side(result)
    _add_segment_shading(ax, result['sides'][dom]['segments'], fs=fs,
                         hatch=None, side_tag=None, seen_labels=seen,
                         frame_range=frame_range)


def _shade_song_legacy(ax, result, fs, seen, frame_range=None):
    # Hatched overlay of the legacy detector's segments (if present) so a
    # reviewer can eyeball new-vs-legacy agreement on any figure that already
    # calls _shade_song. Safe to call when _legacy keys are absent.
    dom = _dom_side(result)
    segs = result['sides'][dom].get('segments_legacy')
    if not segs:
        return
    _add_segment_shading(ax, segs, fs=fs, hatch='///', side_tag='legacy',
                         seen_labels=seen, frame_range=frame_range)


In [ ]:
# # --- Figure 2 multi-page PDF: one page per pair ---
# # Generates fig2_song_all_pairs.pdf — same layout as the single-pair Figure 2
# # above, full bout per page, for easy scanning. Reuses the helpers defined in
# # the Figure 2 cell (_shade_song, _add_segment_freq_annotations, etc.), so
# # that cell must have been run first.
# from matplotlib.backends.backend_pdf import PdfPages

# _fs_song = song_cfg.fs


# def _draw_label_track(ax, segments, fs, y_lo, y_hi, tag):
#     """Colored-strip row of segments on ax between y=[y_lo, y_hi]."""
#     ax.text(-0.008, (y_lo + y_hi) / 2, tag,
#             transform=ax.get_yaxis_transform(), ha='right', va='center',
#             fontsize=8, color='#444')
#     for seg in segments or []:
#         stype = seg.get('type')
#         if stype is None or stype == 'quiet':
#             continue
#         s_ms = seg['start'] / fs * 1000.0
#         e_ms = seg['end']   / fs * 1000.0
#         color = _SEG_COLORS.get(stype, '#cccccc33')
#         edge  = _SEG_EDGE_COLORS.get(stype, '#999999')
#         ax.fill_betweenx([y_lo, y_hi], s_ms, e_ms,
#                          facecolor=color, edgecolor=edge,
#                          linewidth=0.6, zorder=3)


# def _build_song_figure_for_pair(result):
#     """Build the Figure-2 layout for one pair over the full bout. Returns fig."""
#     ms = result['song0'] if result['sex']['male_id'] == 'fly0' else result['song1']
#     idx = result['pair_idx']
#     fidx = result.get('filtered_idx')

#     T_ = ms['summary']['n_frames']
#     t_ms_ = np.arange(T_) / _fs_song * 1000.0

#     dom_ = ms['dominant_wing']
#     dom_side_ = ms['sides'][dom_]

#     # Read new- and legacy-detector views explicitly so the comparison plot
#     # is correct regardless of which pipeline populated the primary keys.
#     # With cfg.pipeline='both', primary stays legacy for back-compat — reading
#     # dom_side_['segments'] would silently shade legacy on the "new" track too.
#     seg_new    = dom_side_.get('segments_new',    dom_side_.get('segments', []))
#     wf_new     = dom_side_.get('window_features_new', dom_side_.get('window_features', {}))
#     sum_new    = ms.get('summary_new',    ms.get('summary', {})) or {}
#     seg_legacy = dom_side_.get('segments_legacy')
#     sum_legacy = ms.get('summary_legacy') or {}
#     wf_ = wf_new
#     s_ = sum_new

#     def _shade_new(ax):
#         _add_segment_shading(ax, seg_new, fs=_fs_song, hatch=None,
#                              side_tag=None, seen_labels=set())

#     # Original session/fly id + frame range for this bout.
#     _k_ = result.get('tracker_key0') or result['key0']
#     _j_ = bout_keys.index(_k_)
#     _fid_ = info.get('fly_ids', [None] * (_j_ + 1))[_j_]
#     _fid_str_ = (_fid_.decode() if isinstance(_fid_, (bytes, bytearray))
#                  else str(_fid_) if _fid_ is not None else '?')
#     _sf_ = int(info.get('start_frames', [0] * (_j_ + 1))[_j_])
#     _ef_ = int(info.get('end_frames',   [0] * (_j_ + 1))[_j_])

#     joints_ = ms.get('joints')
#     # Comparison row drawn only when BOTH detectors actually ran (i.e.
#     # cfg.pipeline='both'). Single-pipeline runs skip the comparison row.
#     has_both = ('segments_new' in dom_side_) and (seg_legacy is not None)
#     base_rows = 3 if joints_ is not None else 2
#     nrows_ = base_rows + (1 if has_both else 0)
#     heights_ = [1.2, 1.0] + ([1.2] if joints_ is not None else [])
#     if has_both:
#         heights_ = heights_ + [0.45]
#     fig_, axes_ = plt.subplots(
#         nrows_, 1, figsize=(10.0, 3.0 * base_rows + (0.9 if has_both else 0)),
#         sharex=True,
#         gridspec_kw={'height_ratios': heights_, 'hspace': 0.28},
#     )
#     if nrows_ == 1:
#         axes_ = [axes_]
#     _pair_label = (f'filtered_idx {fidx} (orig pair_idx {idx})'
#                    if fidx is not None else f'pair {idx}')
#     _frac_line = (f"new: pulse={sum_new.get('frac_pulse', 0):.2f} "
#                   f"sine={sum_new.get('frac_sine', 0):.2f}"
#                   + (f"  |  legacy: pulse={sum_legacy.get('frac_pulse', 0):.2f} "
#                      f"sine={sum_legacy.get('frac_sine', 0):.2f} "
#                      f"waggle={sum_legacy.get('frac_waggle', 0):.2f}"
#                      if sum_legacy else ''))
#     fig_.suptitle(
#         f'{_pair_label}  |  {_fid_str_}  |  frames {_sf_}–{_ef_}  |  '
#         f'({result["key0"]}/{result["key1"]})  |  '
#         f'{ms["summary"]["duration_s"]:.3f}s  |  male={result["sex"]["male_id"]}\n'
#         f'{_frac_line}',
#         fontsize=9, fontweight='bold', y=0.995,
#     )

#     wd_ = ms['wing_data']

#     # Panel 1: Wing tip Z
#     ax = axes_[0]
#     _shade_new(ax)
#     ax.plot(t_ms_, np.asarray(wd_['WingL_V13']['z']),
#             color=_WING_COLORS['WingL_V13'], lw=1.2, alpha=0.9, label='L V13')
#     ax.plot(t_ms_, np.asarray(wd_['WingR_V13']['z']),
#             color=_WING_COLORS['WingR_V13'], lw=1.2, alpha=0.9, label='R V13')
#     _add_segment_freq_annotations(ax, seg_new, wf_new, _fs_song)
#     ax.set_ylabel('Z (world)')
#     ax.set_title('Wing tip Z (V13) — shading: new detector')
#     ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=7, ncol=1)
#     ax.grid(True, alpha=0.3)

#     # Panel 2: Wing tip Y
#     ax = axes_[1]
#     _shade_new(ax)
#     ax.plot(t_ms_, np.asarray(wd_['WingL_V13']['y']),
#             color=_WING_COLORS['WingL_V13'], lw=1.2, alpha=0.9, label='L V13')
#     ax.plot(t_ms_, np.asarray(wd_['WingR_V13']['y']),
#             color=_WING_COLORS['WingR_V13'], lw=1.2, alpha=0.9, label='R V13')
#     ax.set_ylabel('Y (world)')
#     ax.set_title('Wing tip Y (V13)')
#     ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=7, ncol=1)
#     ax.grid(True, alpha=0.3)

#     # Panel 3 (optional): qpos wing joint angles
#     if joints_ is not None:
#         ax = axes_[2]
#         _shade_new(ax)
#         _joint_styles = [
#             ('yaw_L',   '#7B2D8E', '-',  'L yaw'),
#             ('roll_L',  '#7B2D8E', '--', 'L roll'),
#             ('pitch_L', '#7B2D8E', ':',  'L pitch'),
#             ('yaw_R',   '#0369A1', '-',  'R yaw'),
#             ('roll_R',  '#0369A1', '--', 'R roll'),
#             ('pitch_R', '#0369A1', ':',  'R pitch'),
#         ]
#         for jkey, jcolor, jls, jlabel in _joint_styles:
#             if jkey in joints_:
#                 ax.plot(t_ms_, np.degrees(np.asarray(joints_[jkey])),
#                         color=jcolor, ls=jls, lw=1.0, alpha=0.85, label=jlabel)
#         ax.set_ylabel('angle (deg)')
#         ax.set_title('Wing joint angles (qpos)')
#         ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=7, ncol=2)
#         ax.grid(True, alpha=0.3)

#     # Label-comparison track (new on top, legacy on bottom). Only drawn when
#     # both detectors actually ran (cfg.pipeline='both').
#     if has_both:
#         lab_ax = axes_[-1]
#         _draw_label_track(lab_ax, seg_new,
#                           _fs_song, 0.55, 0.95, 'new')
#         _draw_label_track(lab_ax, seg_legacy or [],
#                           _fs_song, 0.05, 0.45, 'legacy')
#         lab_ax.set_ylim(0, 1)
#         lab_ax.set_yticks([])
#         lab_ax.set_title('new vs legacy song labels', fontsize=9)
#         lab_ax.grid(True, axis='x', alpha=0.3)

#     axes_[-1].set_xlabel('time (ms)')

#     # Pulse event overlay (new detector). wf_new is the new features dict.
#     _pef = np.asarray(wf_new.get('pulse_event_frames', np.array([], dtype=int)))
#     if _pef.size > 0:
#         _pef_ms = _pef / _fs_song * 1000.0
#         # Only overlay on signal panels, not on the label-track row.
#         signal_axes = axes_[:base_rows]
#         for _ax in signal_axes:
#             for i_, xm_ in enumerate(_pef_ms):
#                 _ax.axvline(xm_, color='#E76F51', lw=0.6, alpha=0.55,
#                             ls='--', zorder=4,
#                             label=(f'pulse events (n={len(_pef_ms)})'
#                                    if i_ == 0 and _ax is signal_axes[0] else None))
#         signal_axes[0].legend(loc='upper left', bbox_to_anchor=(1.01, 1),
#                               fontsize=7)

#     return fig_


# _out_pdf = FIG_DIR / 'fig2_song_all_pairs.pdf'
# _n = len(results)
# print(f'writing {_n} pair figures to {_out_pdf}')
# with PdfPages(_out_pdf) as _pdf:
#     for _i, _r in enumerate(results):
#         try:
#             _fig = _build_song_figure_for_pair(_r)
#             _pdf.savefig(_fig, bbox_inches='tight')
#             plt.close(_fig)
#         except Exception as _e:
#             print(f'  pair {_r.get("pair_idx", _i)}: {type(_e).__name__}: {_e}')
#         if (_i + 1) % 25 == 0 or (_i + 1) == _n:
#             print(f'  wrote {_i + 1}/{_n}')
# print(f'saved {_out_pdf}')


# Figure 4 — Courtship Figure

Combines the EXAMPLE_IDX=12 exemplar bout (rows 1–3) with population summaries
(row 4). Layout is built by `utils.courtship_figure_panels.assemble_figure` and
each panel is filled by an independent `panel_*` function so any panel can be
swapped out without touching the others.

Fill in the three placeholders below before running:
- `CAM_MP4_PATH` — path to the per-camera mp4 covering this bout
- `ROI`         — (x, y, w, h) crop in pixels
- `FREE_WALK_H5_PATH` — path to a non-courtship free-walking combined h5


In [ ]:
# --- Figure 4: Reorganized consolidated courtship figure (183 mm × 150 mm) ---
import importlib
import sys
import numpy as np
import matplotlib.pyplot as plt

from utils import courtship_figure_panels as cfp
from utils.sam3_female_com import triangulate_sam3_female_com
from utils.sam3_female_com import sam3_camera_index, unpack_sam3_masks_for_frames
from utils.sam3_aligned_bouts import compute_per_bout_pitch_alignment
from utils import courtship_loader as _cl
from utils import pulse_type_cache as _ptc
from utils import free_walking_loader as _fwl

importlib.reload(cfp)
importlib.reload(_cl)
importlib.reload(_ptc)
importlib.reload(_fwl)
get_pulse_type_labels = _ptc.get_pulse_type_labels
load_free_walking_scutellum_z = _fwl.load_free_walking_scutellum_z
load_orig_keypoints_index = _cl.load_orig_keypoints_index
get_orig_keypoints_for_combined_bout = _cl.get_orig_keypoints_for_combined_bout

# ------------------------------------------------------------------ inputs
# --- Video (rows 1+2) -------------------------------------------------
# CAM_MP4_PATH  = "/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/Cam2012630.mp4"
# SESSION_DIR = Path('/data2/users/eabe/datasets/Johnson_lab/courtship/Session1/Session1_bouts_04172026/2026_04_02_16_21_32/')
# SESSION_DIR = Path('/data2/users/eabe/datasets/Johnson_lab/courtship/Session1/Session1_bouts_04172026/2026_04_02_16_21_32/')
# CAM_MP4_PATH  = SESSION_DIR/"Cam2012630.mp4"
# DLT_CSV_PATH  = SESSION_DIR/"calibration/Cam2012630_dlt.csv"
CAM_MP4_PATH      = "/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/Cam2012630.mp4"
DLT_CSV_PATH      = "/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/calibration/Cam2012630_dlt.csv"
VIDEO_ROI         = (1350, 0, 500, 500)             # (x, y, w, h) static crop in raw video pixels
DATA3D_FLY0_CSV   = '/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/Predictions_3D_V4_phase4_enumfix/bout_00028/fly1.csv'
DATA3D_FLY1_CSV   = '/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/Predictions_3D_V4_phase4_enumfix/bout_00028/fly0.csv'
# DATA3D_FLY0_CSV   = SESSION_DIR / 'Predictions_3D_34662592/bout_00006/fly0.csv'
# DATA3D_FLY1_CSV   = SESSION_DIR / 'Predictions_3D_34662592/bout_00006/fly1.csv'
KP_LABEL_FRAME    = None                            # None = middle of strip
KP_LABEL_OFFSETS  = {                               # text nudge in cropped px
    "Scutellum":  (0,0),#( 8, -4),
    "WingL_V13":  (0,0),#(-12, -10),
    "WingR_V13":  (0,0),#( 12, -10),
}
KP_SCALE         = 0.1                              # 3D unit -> DLT unit (mm -> cm here)
SHOW_FEMALE_KP   = False                            # Panel B: show fly1 (female) keypoints?
SAM3_MASK_PATH = '/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/Predictions_3D_V4_phase4_enumfix/bout_00028/sam3_masks.npz'
CALIB_DIR      = '/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/calibration'
SAM3_ALIGNED_H5 = '/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/Predictions_3D_34662304/sam3_aligned.h5'
BOUTS_ROOT      = '/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/Predictions_3D_34662304'
# SAM3_MASK_PATH = SESSION_DIR /'Predictions_3D_34662592/bout_00006/sam3_masks.npz'
# CALIB_DIR      = SESSION_DIR / 'calibration'
# SAM3_ALIGNED_H5 = SESSION_DIR / 'Predictions_3D_34662592/sam3_aligned.h5'
# BOUTS_ROOT      = SESSION_DIR / 'Predictions_3D_34662592'

# Original (pre-rescale) keypoints — required for correct DLT projection.
# Combined h5 stores the body-model-rescaled kp_data; orig_keypoints live
# in each per-prediction `preprocessing/preprocessed_bout_*_both.h5`.
ORIG_KP_SEARCH_DIR = "/data2/users/eabe/datasets/Johnson_lab/courtship/04092026_bouts_04112026"
# ORIG_KP_SEARCH_DIR = SESSION_DIR / "Predictions_3D_34662592"

# --- Free-walking dataset (row 3 right) --------------------------------
FREE_WALK_H5_PATH = "/data2/users/eabe/datasets/Johnson_lab/free_walking/Data_analysis/analysis/v1/ik_output_combined_v1_free_walking.h5"

# --- Render (row 4): male + female + floor -----------------------------
# fruitfly_v1 is vendored under models/fruitfly_v1/ in the repo.
_REPO_ROOT_VIZ = Path(globals().get("__file__", Path.cwd() / "_")).resolve().parent.parent if "__file__" in globals() else Path.cwd().parent
VIZ_MODEL_XML    = str(_REPO_ROOT_VIZ / "models" / "fruitfly_v1" / "fruitfly_v1_free.xml")
VIZ_FLOOR_XML    = str(_REPO_ROOT_VIZ / "models" / "fruitfly_v1" / "floor.xml")
VIZ_SETTINGS_FLY0 = "Earthy_V1_courtship_fly0"
VIZ_SETTINGS_FLY1 = "Earthy_V1_courtship_fly1"
VIZ_CAMERA       = "track1_fly0"
VIZ_HEIGHT       = 512
VIZ_WIDTH        = 512
RENDER_CROP_WH   = (320, 320)                      # (w, h) centered post-render crop; wider so fly1 stays in frame
RENDER_CAM_DISTANCE   = 1.75                        # midpoint-camera distance (meters)
RENDER_CAM_AZIMUTH    = 85.0                        # midpoint-camera azimuth (deg)
RENDER_CAM_ELEVATION  = -20.0                       # midpoint-camera elevation (deg)
RENDER_CROP      = (VIZ_WIDTH // 2 - RENDER_CROP_WH[0] // 2,
                    VIZ_HEIGHT // 2 - RENDER_CROP_WH[1] // 2,
                    RENDER_CROP_WH[0], RENDER_CROP_WH[1])

# --- Exemplar bout + frame selection -----------------------------------
# Locate the Session0/2025_10_20_13_20_04 exemplar by fly_id + start_frame
# within the post-filter `results` list. `pair_bouts` can drop some pairs
# (short bouts, no bilateral wing activity) so the list index is not
# identical to the raw pair index from `pair_bouts`.
_TARGET_FLY_ID_PREFIX = 'Session0/2025_10_20_13_20_04'
_TARGET_START_FRAME   = 446306
# _TARGET_FLY_ID_PREFIX = 'Session1_bouts_04172026/2026_04_02_16_21_32'
# _TARGET_START_FRAME   = 505035
_fly_ids_list     = list(info.get('fly_ids', []))
_start_frames_list = list(info.get('start_frames', []))
EXAMPLE_IDX = None
for _i, _r in enumerate(results):
    _j = bout_keys.index(_r['tracker_key0'])
    _fid = _fly_ids_list[_j]
    _fid_s = _fid.decode() if isinstance(_fid, bytes) else str(_fid)
    if (_TARGET_FLY_ID_PREFIX in _fid_s
            and int(_start_frames_list[_j]) == _TARGET_START_FRAME):
        EXAMPLE_IDX = _i
        break
if EXAMPLE_IDX is None:
    raise RuntimeError(f'exemplar not found: '
                       f'{_TARGET_FLY_ID_PREFIX} @ {_TARGET_START_FRAME}')
print(f'[exemplar] results[{EXAMPLE_IDX}]  pair_idx='
      f'{results[EXAMPLE_IDX]["pair_idx"]}  '
      f'T={int(results[EXAMPLE_IDX]["T"])}')
START_FRAME     = int((0   / 1000) * 800)
END_FRAME       = None # int((800 / 1000) * 800)           # None => full clip
N_FRAMES_STRIP  = 6
N_RENDER_STRIP  = 4
FRAME_INDICES   = None                              # None = evenly spaced

# ------------------------------------------------------------------ slice
fs = song_cfg.fs
ex = results[EXAMPLE_IDX]

clip_T = int(ex["T"])
if END_FRAME is None:
    END_FRAME = clip_T
END_FRAME = min(int(END_FRAME), clip_T)
START_FRAME = max(0, int(START_FRAME))
sel = slice(START_FRAME, END_FRAME)

wing_data   = ex["song0"]["wing_data"]
wingL_z     = wing_data["WingL_V13"]["z"][sel]
wingR_z     = wing_data["WingR_V13"]["z"][sel]
scutellum_z = ex["com_z"][sel]
seg_L       = ex["song0"]["sides"]["L"]["segments"]
seg_R       = ex["song0"]["sides"]["R"]["segments"]
t_ms        = (np.arange(START_FRAME, END_FRAME) / fs) * 1000.0
if FRAME_INDICES is not None:
    frame_idx_strip = np.asarray(FRAME_INDICES, dtype=int)
else:
    frame_idx_strip = np.linspace(START_FRAME, END_FRAME - 1,
                                  N_FRAMES_STRIP, dtype=int)
N_FRAMES_STRIP = len(frame_idx_strip)

# Per-frame extended vs folded wing for the exemplar bout (Row 3 left).
angle_L = np.asarray(ex["song0"]["angle_L"])
angle_R = np.asarray(ex["song0"]["angle_R"])
zL_full = np.asarray(wing_data["WingL_V13"]["z"])
zR_full = np.asarray(wing_data["WingR_V13"]["z"])
ext_is_L = angle_L > angle_R
ext_z_w  = np.where(ext_is_L, zL_full, zR_full)[sel]
fold_z_w = np.where(ext_is_L, zR_full, zL_full)[sel]

# ------------------------------------------------------------------ aggregates
# Per-bout MEAN scutellum z by song state (Row 3 right; n = bouts, not frames).
pulse_z_list, sine_z_list = [], []
for r in results:
    v = np.asarray(r["male_valid"], dtype=bool)
    lab = np.asarray(r["male_labels"])
    z = np.asarray(r["com_z"], dtype=float)
    base = v & np.isfinite(z)
    mp = base & (lab == "pulse")
    ms = base & (lab == "sine")
    if mp.any():
        pulse_z_list.append(float(np.mean(z[mp])))
    if ms.any():
        sine_z_list.append(float(np.mean(z[ms])))
pulse_z = np.asarray(pulse_z_list, dtype=float)
sine_z  = np.asarray(sine_z_list,  dtype=float)

try:
    walking_z = load_free_walking_scutellum_z(FREE_WALK_H5_PATH, per_bout=True)
except (FileNotFoundError, OSError) as e:
    print(f"(free walking h5 not loaded — {e}); using empty array")
    walking_z = np.zeros(0)

# Pooled extended-wing horizontal angles by song state (Row 3 col 2).
# Horizontal angle = angle in the body's horizontal plane between the
# fore-aft body axis and the wing vector (0 deg rest, 90 deg fully
# extended). Uses compute_wing_horizontal_angles so the angle is truly
# horizontal regardless of the MuJoCo joint axis orientation.
ext_pulse_a, ext_sine_a = [], []
for r in results:
    hL = r["song0"].get("horiz_angle_L")
    hR = r["song0"].get("horiz_angle_R")
    if hL is None or hR is None:
        continue
    yL = np.asarray(hL, dtype=float)
    yR = np.asarray(hR, dtype=float)
    lab = np.asarray(r["male_labels"])
    v   = np.asarray(r["male_valid"], dtype=bool)
    ext_is_L_r = np.abs(yL) > np.abs(yR)
    ext_a = np.abs(np.where(ext_is_L_r, yL, yR))
    base = v & np.isfinite(ext_a)
    mp = base & (lab == "pulse")
    ms = base & (lab == "sine")
    if mp.any(): ext_pulse_a.append(ext_a[mp])
    if ms.any(): ext_sine_a.append(ext_a[ms])
ext_pulse_a = np.concatenate(ext_pulse_a) if ext_pulse_a else np.zeros(0)
ext_sine_a  = np.concatenate(ext_sine_a)  if ext_sine_a  else np.zeros(0)

# Pooled L vs R wing-z phase difference during sine song (Row 3 col 0 polar).
from scipy.signal import hilbert as _hilbert
phase_diffs = []
for r in results:
    wd = r["song0"]["wing_data"]
    zL_full_r = np.asarray(wd["WingL_V13"]["z"], dtype=float)
    zR_full_r = np.asarray(wd["WingR_V13"]["z"], dtype=float)
    for s in r["song0"]["sides"]["L"]["segments"]:
        if s.get("type") != "sine":
            continue
        i0, i1 = int(s["start"]), int(s["end"]) + 1
        if i1 - i0 < 16:
            continue
        a = zL_full_r[i0:i1]; b = zR_full_r[i0:i1]
        if not (np.all(np.isfinite(a)) and np.all(np.isfinite(b))):
            continue
        a = a - np.mean(a); b = b - np.mean(b)
        pa = np.angle(_hilbert(a))
        pb = np.angle(_hilbert(b))
        R_seg = np.mean(np.exp(1j * (pa - pb)))
        if not np.isfinite(R_seg):
            continue
        phase_diffs.append(np.angle(R_seg))
phase_diffs = (np.asarray(phase_diffs, dtype=float) if phase_diffs
               else np.zeros(0, dtype=float))

# Pulse-type labels (cached) for wing/scut shading.
pulse_type_results = get_pulse_type_labels(
    results, cache_path=FIG_DIR / "pulse_type_cache.pkl", fs=fs
)
pulse_type_labels = pulse_type_results["labels"]

_ex_pair = int(ex.get("pair_idx", EXAMPLE_IDX))
pulse_type_side = {}
for _side in ("L", "R"):
    _pf = ex["song0"]["sides"].get(_side, {}).get("pulse_features", {}) or {}
    _peaks = np.asarray(_pf.get("peak_frames", np.zeros(0, dtype=int)))
    _labs  = np.asarray(pulse_type_labels.get(_ex_pair, {}).get(_side, np.array([])))
    if _peaks.size and _labs.size == _peaks.size:
        pulse_type_side[_side] = {"peak_frames": _peaks, "labels": _labs}

_dw = str(ex["song0"].get("dominant_wing", "L")).upper()
_dom_side = "L" if _dw.startswith("L") else "R"
_dom_track = pulse_type_side.get(_dom_side, {})

# DLT calibration + ORIGINAL 3D keypoints (un-rescaled) for projection.
dlt_coeffs = cfp._dlt_load(DLT_CSV_PATH)
VIDEO_FRAME_OFFSET = int(info["start_frames"][bout_keys.index(ex["key0"])])
try:
    orig_kp_index = load_orig_keypoints_index(ORIG_KP_SEARCH_DIR)
except (FileNotFoundError, OSError) as e:
    print(f"(orig keypoints not loaded — {e}); falling back to body-model kp_data")
    orig_kp_index = {}
kp_xyz_fly0 = get_orig_keypoints_for_combined_bout(
    orig_kp_index, info, bout_keys, ex["key0"]
)
kp_xyz_fly1 = get_orig_keypoints_for_combined_bout(
    orig_kp_index, info, bout_keys, ex["key1"]
)
_USE_KP_OVERLAY = kp_xyz_fly0 is not None
if kp_xyz_fly0 is None:
    print("(no orig_keypoints match for fly0 — keypoint overlays disabled)")
    kp_xyz = data[ex["key0"]]["kp_data"]
else:
    kp_xyz = kp_xyz_fly0
    print(f"orig_keypoints fly0: {kp_xyz_fly0.shape}  "
          f"fly1: {None if kp_xyz_fly1 is None else kp_xyz_fly1.shape}")

# Per-frame midpoint of the two flies' Scutellum in the DLT world frame.
# orig_keypoints are stored per-fly (T, N, 3) with N = len(kp_names), so we
# must load BOTH fly0 and fly1 arrays and midpoint them. Fall back to fly0
# alone when fly1 is missing, and to kp_data (re-centered, less accurate)
# when no orig_keypoints are available.
_scut_idx = kp_names.index("Scutellum")
if kp_xyz_fly0 is not None and kp_xyz_fly1 is not None:
    _T_c = min(kp_xyz_fly0.shape[0], kp_xyz_fly1.shape[0], clip_T)
    center_xyz = 0.5 * (
        kp_xyz_fly0[:_T_c, _scut_idx, :] + kp_xyz_fly1[:_T_c, _scut_idx, :]
    )
    print(f"center_xyz from orig fly0+fly1 midpoint: shape {center_xyz.shape}")
elif kp_xyz_fly0 is not None:
    center_xyz = kp_xyz_fly0[:, _scut_idx, :]
    print(f"center_xyz from orig fly0 only: shape {center_xyz.shape}")
else:
    _kp0 = np.asarray(data[ex["key0"]]["kp_data"])
    _kp1 = np.asarray(data[ex["key1"]]["kp_data"])
    if _kp0.ndim == 2 and _kp0.shape[-1] != 3:
        _kp0 = _kp0.reshape(_kp0.shape[0], -1, 3)
        _kp1 = _kp1.reshape(_kp1.shape[0], -1, 3)
    _T_c = min(_kp0.shape[0], _kp1.shape[0], clip_T)
    center_xyz = 0.5 * (_kp0[:_T_c, _scut_idx, :] + _kp1[:_T_c, _scut_idx, :])
    print(f"center_xyz from kp_data fallback: shape {center_xyz.shape}")

# Sanity-project center_xyz[0] so we can see if it lands inside the video.
try:
    _uv0 = cfp._dlt_project(dlt_coeffs, center_xyz[0] * KP_SCALE)
    print(f"center_xyz[0] -> uv {_uv0} (expected 0..~512 if DLT/scale OK)")
except Exception as _e:
    print(f"(DLT sanity-project failed: {_e})")


# --- Raw JARVIS data3D_*.csv override -------------------------------
# When the per-session raw CSVs are available, use them directly: they
# are in the true DLT world frame (confirmed by the diagnostic cell),
# span the full session, and contain both flies separately. Reorders CSV
# columns into the combined-h5 `kp_names` ordering.
def _load_data3d_csv(path, kp_names_order):
    if not path:
        return None, None
    import pandas as _pd
    df = _pd.read_csv(path, header=[0, 1])
    names_top = [c[0] for c in df.columns]
    axes      = [c[1] for c in df.columns]
    arr = df.to_numpy(dtype=float)
    out = np.full((arr.shape[0], len(kp_names_order), 4), np.nan, dtype=float)
    name_to_out = {n: i for i, n in enumerate(kp_names_order)}
    axis_map = {'x': 0, 'y': 1, 'z': 2, 'confidence': 3}
    for ci, (nm, ax) in enumerate(zip(names_top, axes)):
        oi = name_to_out.get(nm)
        aj = axis_map.get(str(ax).lower())
        if oi is None or aj is None:
            continue
        out[:, oi, aj] = arr[:, ci]
    return out[..., :3], out[..., 3]

if DATA3D_FLY0_CSV or DATA3D_FLY1_CSV:
    _xyz0, _ = _load_data3d_csv(DATA3D_FLY0_CSV, kp_names)
    _xyz1, _ = _load_data3d_csv(DATA3D_FLY1_CSV, kp_names)
    if _xyz0 is not None:
        kp_xyz_fly0 = _xyz0; kp_xyz = _xyz0
    if _xyz1 is not None:
        kp_xyz_fly1 = _xyz1
    _USE_KP_OVERLAY = kp_xyz_fly0 is not None
    if kp_xyz_fly0 is not None and kp_xyz_fly1 is not None:
        _T_c = min(kp_xyz_fly0.shape[0], kp_xyz_fly1.shape[0], clip_T)
        center_xyz = 0.5 * (
            kp_xyz_fly0[:_T_c, _scut_idx, :] + kp_xyz_fly1[:_T_c, _scut_idx, :]
        )
    print(f'[csv override] fly0={None if _xyz0 is None else _xyz0.shape}  '
          f'fly1={None if _xyz1 is None else _xyz1.shape}')

    # Clamp the trace/strip frame range to whatever the CSV override
    # supports; combined-h5 ex["T"] may extend past the raw CSV length.
    _csv_T = None
    if kp_xyz_fly0 is not None:
        _csv_T = int(kp_xyz_fly0.shape[0])
        if kp_xyz_fly1 is not None:
            _csv_T = min(_csv_T, int(kp_xyz_fly1.shape[0]))
    if _csv_T is not None and _csv_T < END_FRAME:
        END_FRAME = _csv_T
        sel = slice(START_FRAME, END_FRAME)
        wingL_z     = wing_data["WingL_V13"]["z"][sel]
        wingR_z     = wing_data["WingR_V13"]["z"][sel]
        scutellum_z = ex["com_z"][sel]
        ext_z_w     = np.where(ext_is_L, zL_full, zR_full)[sel]
        fold_z_w    = np.where(ext_is_L, zR_full, zL_full)[sel]
        t_ms        = (np.arange(START_FRAME, END_FRAME) / fs) * 1000.0
        frame_idx_strip = np.linspace(
            START_FRAME, END_FRAME - 1, N_FRAMES_STRIP, dtype=int,
        )
        print(f'[csv override] clamped END_FRAME to {END_FRAME} '
              f'(CSV len = {_csv_T}); frame_idx_strip rebuilt')

# Panel J: male body pitch + target pitch over the FULL bout (from CSV override),
# not the trimmed ex["T"] h5 clip.
female_com_world = triangulate_sam3_female_com(
    SAM3_MASK_PATH, CALIB_DIR, fly_idx=0, min_cams=2, verbose=False,
)
# Use the full CSV length for the pitch trace rather than ex["T"]=clip_T.
clip_T_full = int(kp_xyz_fly0.shape[0])
female_com_bout_full = female_com_world[:clip_T_full] / KP_SCALE

_head_idx = kp_names.index("Antenna_Base")
head_xyz_full = kp_xyz_fly0[:clip_T_full, _head_idx, :]
scut_xyz_full = kp_xyz_fly0[:clip_T_full, _scut_idx, :]

# Body pitch from the male thorax quaternion (qpos[3:7], MuJoCo [w,x,y,z]).
# Local +X is anterior, so world-frame forward z = 2*(qx*qz - qw*qy).
# qpos may be 1 frame shorter than kp CSVs; clip to min length.
_qpos_male_full = np.asarray(data[ex["key0"]].get("qpos"))
clip_T_full = int(min(clip_T_full, _qpos_male_full.shape[0]))
female_com_bout_full = female_com_bout_full[:clip_T_full]
head_xyz_full        = head_xyz_full[:clip_T_full]
scut_xyz_full        = scut_xyz_full[:clip_T_full]
male_body_pitch_full = cfp.body_pitch_deg_from_quat(
    _qpos_male_full[:clip_T_full, 3:7])

target_vec_full = female_com_bout_full - scut_xyz_full
_tg_norm_full = np.linalg.norm(target_vec_full, axis=-1)
target_pitch_full = np.degrees(np.arcsin(np.divide(
    target_vec_full[..., 2], _tg_norm_full,
    out=np.full_like(_tg_norm_full, np.nan), where=_tg_norm_full > 0)))

pitch_alignment_full = male_body_pitch_full - target_pitch_full
t_ms_full = (np.arange(clip_T_full) / fs) * 1000.0

# Panel J: per-bout pitch alignment POOLED across all courtship sessions
# (globs every Predictions_3D_*/sam3_aligned.h5 under courtship_root).
from utils.sam3_aligned_bouts import (
    compute_pitch_alignment_all_sessions, find_courtship_sessions,
)
COURTSHIP_ROOT = Path('/data2/users/eabe/datasets/Johnson_lab/courtship')
_all_sessions = find_courtship_sessions(COURTSHIP_ROOT)
print(f'[pitch-violin] {len(_all_sessions)} sessions discovered')
_all_align = compute_pitch_alignment_all_sessions(
    _all_sessions, kp_scale=KP_SCALE,
)
per_bout_align = _all_align['median_abs_alignment_deg']
if _all_align['skipped']:
    for _p, _err in _all_align['skipped']:
        print(f'[pitch-violin] skipped {_p}: {_err}')
print(f'[pitch-violin] {per_bout_align.size} bouts pooled across '
      f'{len(_all_align["session_paths"])} sessions')
# Locate the exemplar in the pooled vector: exemplar session = the
# Predictions_3D_* folder we used for the single-bout panels, and
# exemplar bout = bout_00006 inside it.
_exemplar_session = str(BOUTS_ROOT)
_exemplar_bout = 'bout_00006'
EXEMPLAR_BOUT_IDX = None
_sp = _all_align['session_paths']
_sob = _all_align['session_of_bout']
_names = _all_align['bout_names_global']
if _exemplar_session in _sp:
    _sid = _sp.index(_exemplar_session)
    for _gi, (_s, _n) in enumerate(zip(_sob, _names)):
        if int(_s) == _sid and _n == _exemplar_bout:
            EXEMPLAR_BOUT_IDX = _gi
            break
print(f'[pitch-violin] exemplar global idx = {EXEMPLAR_BOUT_IDX}')


# ------------------------------------------------------------------ assemble
fig, axd = cfp.assemble_figure(fig_width_mm=183.0, fig_height_mm=140.0,
                               n_frames_strip=N_FRAMES_STRIP, n_render_strip=N_RENDER_STRIP,
                               n_video_frames=4)

# Row 1 right — wing V13 z + scutellum z time series
# Drop waggle segments so Wing V13 z shading stays restricted to pulse/sine.
_seg_L_wing = [s for s in seg_L if s.get('type') != 'waggle']
_seg_R_wing = [s for s in seg_R if s.get('type') != 'waggle']
cfp.panel_wing_z_traces(axd["wing"], t_ms, wingL_z, wingR_z,
                        _seg_L_wing, _seg_R_wing,
                        fs=fs, frame_range=(START_FRAME, END_FRAME),
                        pulse_type_side=pulse_type_side,
                        min_segment_ms=10.0, time_unit='s')
# Filter out spurious sub-10ms segments (matches Panel B wing shading)
# and drop pulse-subtype coloring so pulse segments read as the default
# pulse color rather than Pslow/Pfast-specific shades.
_min_frames_scut = max(1, int(round(10.0 * fs / 1000.0)))
_seg_L_scut = [s for s in seg_L
               if (int(s["end"]) - int(s["start"])) >= _min_frames_scut]
cfp.panel_scutellum_z_trace(axd["scut"], t_ms, scutellum_z,
                            segments=_seg_L_scut, fs=fs,
                            frame_range=(START_FRAME, END_FRAME), time_unit='s')

# Row 2 — video frames with all keypoints overlayed
try:
    if _USE_KP_OVERLAY:
        # SAM3 segmentation masks overlayed on the video strip.
        # SAM3 fly_idx=0 is the female, fly_idx=1 is the male; Panel C
        # draws masks aligned to the kp_xyz_per_frame / kp_xyz_fly1_per_frame
        # ordering (male, female) so colors match the KP dots.
        _cam_csv_name = Path(DLT_CSV_PATH).name
        _sam3_cam_idx = sam3_camera_index(CALIB_DIR, _cam_csv_name)
        _frame_idx_strip_C = frame_idx_strip[: len(axd["video"])]
        _sam3_masks = unpack_sam3_masks_for_frames(
            SAM3_MASK_PATH, _sam3_cam_idx,
            fly_indices=[1, 0],
            frame_indices=[int(f) for f in _frame_idx_strip_C],
        )
        cfp.panel_video_strip_with_kp(
            axd["video"], CAM_MP4_PATH, _frame_idx_strip_C,
            kp_xyz_per_frame=kp_xyz_fly0, kp_names=kp_names, dlt_coeffs=dlt_coeffs,
            kp_xyz_fly1_per_frame=(kp_xyz_fly1 if SHOW_FEMALE_KP else None),
            kp_color='#e74c3c', kp_color_fly1='#3a7bff',
            overlay_kps=None, roi=VIDEO_ROI, fs=fs,
            kp_scale=KP_SCALE, video_frame_offset=VIDEO_FRAME_OFFSET,
            masks_per_fly=_sam3_masks,
            mask_colors=['#e74c3c', '#3a7bff'],
            mask_alpha=0.35,
        )
    else:
        cfp.panel_video_strip(
            axd["video"], CAM_MP4_PATH, frame_idx_strip[: len(axd["video"])],
            roi=VIDEO_ROI, fs=fs,
            video_frame_offset=VIDEO_FRAME_OFFSET,
        )
except (FileNotFoundError, OSError) as e:
    print(f"(video strip not loaded — {e}); leaving Row 2 blank")

# Row 3 — sine in-phase | joint angle density | Pslow/Pfast | scutellum z
sine_segs_window = [s for s in seg_L if s.get("type") == "sine"]

# Zoom window for the sine in-phase panel (ms, relative to bout start).
SINE_PHASE_T_MS = (800.0, 1100.0)
_zf0 = max(START_FRAME, int(round(SINE_PHASE_T_MS[0] * fs / 1000.0)))
_zf1 = min(END_FRAME,   int(round(SINE_PHASE_T_MS[1] * fs / 1000.0)))
_zsel = slice(_zf0 - START_FRAME, _zf1 - START_FRAME)
# Shift sine-phase panel input so its x-axis starts at 0 ms.
_sine_segs_shift = [
    {**s, 'start': int(s['start']) - _zf0, 'end': int(s['end']) - _zf0}
    for s in sine_segs_window if int(s['end']) > _zf0 and int(s['start']) < _zf1
]
cfp.panel_sine_wing_inphase(
    axd["sine_phase"], t_ms[_zsel] - t_ms[_zsel][0], ext_z_w[_zsel], fold_z_w[_zsel], fs=fs,
    frame_range=(0, _zf1 - _zf0),
    sine_segments=_sine_segs_shift, title=''
)
cfp.panel_wing_phase_polar(
    axd["wing_phase_polar"], phase_diffs, center_stat='median',
    title='',
)
cfp.panel_joint_angle_density(
    axd["angle_2d"], ext_pulse_a, ext_sine_a,
    range_deg=(0.0, 90.0),
    title="",
)
cfp.panel_pulse_classification(axd["pulse_class"], pulse_type_results,
                               show_std=True, title='')
cfp.panel_z_height_singing_vs_walking(axd["zheight"], pulse_z, sine_z, walking_z,
                                      kind="violin", point_kwargs={'s': 3, 'alpha': 0.25}, title='')

# Row 4 — MuJoCo render strip (same frame indices as Row 2)
qpos0 = data[ex["key0"]].get("qpos")
qpos1 = data[ex["key1"]].get("qpos")
if qpos0 is None or qpos1 is None or VIZ_MODEL_XML is None:
    print("(no qpos for one of the flies or no model — leaving Row 4 blank)")
else:
    T_min = min(len(qpos0), len(qpos1), clip_T)
    qpos_pair = np.concatenate(
        [np.asarray(qpos0)[:T_min], np.asarray(qpos1)[:T_min]], axis=-1,
    )
    # Estimate rig pose from all bouts in THIS session (Session0 exemplar).
    # Manual nudge on top; tune to align with chamber walls.
    RIG_CENTER_OFFSET = (0.0, 0.0)
    _session_prefix = _TARGET_FLY_ID_PREFIX
    _session_keys = [
        bout_keys[_i] for _i, _f in enumerate(info.get('fly_ids', []))
        if _session_prefix in (_f.decode() if isinstance(_f, bytes) else str(_f))
    ]
    _rig = cfp.estimate_rig_pose_from_qpos(data, bout_keys=_session_keys)
    _rig_pos = (_rig['center_xy'][0] + RIG_CENTER_OFFSET[0],
                _rig['center_xy'][1] + RIG_CENTER_OFFSET[1])
    print(f"[fig4 rig] using {len(_session_keys)}/{len(bout_keys)} bouts  "
          f"center=({_rig['center_xy'][0]:+.3f}, {_rig['center_xy'][1]:+.3f}) cm  "
          f"extent major={_rig['major_len']:.2f} cm minor={_rig['minor_len']:.2f} cm")
    viz = cfp.build_courtship_pair_visualizer(
        flybody_xml=VIZ_MODEL_XML,
        floor_xml=VIZ_FLOOR_XML,
        settings_fly0=VIZ_SETTINGS_FLY0,
        settings_fly1=VIZ_SETTINGS_FLY1,
        rig_pos=_rig_pos,
    )
    qpos_pair = cfp.floor_align_qpos_pair(viz.model, qpos_pair)
    render_frame_idx = np.linspace(START_FRAME, END_FRAME - 1, N_FRAMES_STRIP, dtype=int)
    cfp.panel_render_strip(
        axd["render"], qpos_pair, render_frame_idx,
        viz=viz, camera=VIZ_CAMERA,
        height_px=VIZ_HEIGHT, width_px=VIZ_WIDTH,
        crop=RENDER_CROP,
        track_midpoint=True,
        cam_distance=RENDER_CAM_DISTANCE,
        cam_azimuth=RENDER_CAM_AZIMUTH,
        cam_elevation=RENDER_CAM_ELEVATION,
    )


cfp.panel_male_pitch(
    axd['pitch'], t_ms_full,
    male_body_pitch_full, target_pitch_full,
    segments=None, fs=fs,
    frame_range=(0, clip_T_full),
    male_color='#d62728', target_color='#1f77b4', time_unit='s',
)
cfp.panel_pitch_alignment_violin(
    axd['align_violin'], per_bout_align,
    exemplar_idx=EXEMPLAR_BOUT_IDX,
)

# plt.tight_layout()
cfp.add_panel_letters(axd, titles={
    # DRAFT titles — edit freely.
    'wing':             'wing z and body height reveal song structure',
    'video':            'example male courtship sequence',
    'sine_phase':       'wings stay in phase during switching',
    'wing_phase_polar': 'L\u2013R wings are in phase',
    'angle_2d':         'pulse and sine wing angles',
    'pulse_class':      'Pslow and Pfast pulse types',
    'zheight':          'body sits lower during song',
    'render':           'reconstructed 3D courtship pose',
    'pitch':            'male pitches up toward female',
    'align_violin':     'body pitch alignment',},
    positions={
            'pitch':        (-0.30, 1.06), 
            # 'align_violin': (-0.60, 1.06), 
               
               })

out_pdf = FIG_DIR / "fig4_042126_ETTA_2.pdf"
out_png = FIG_DIR / "fig4_042126_ETTA_2.png"
out_svg = FIG_DIR / "fig4_042126_ETTA_2.svg"
# fig.savefig(out_png, dpi=300, transparent=True)
# fig.savefig(out_pdf, dpi=300, transparent=True)
# fig.savefig(out_svg, dpi=300,transparent=True)
print(f"saved {out_pdf}")
print(f"saved {out_svg}")
plt.show()

# Panel J sanity check: SAM3 male-COM (fly_idx=1) vs KP-Scutellum (post-override fly0)
try:
    _male_sam3 = triangulate_sam3_female_com(  
        SAM3_MASK_PATH, CALIB_DIR, fly_idx=1, min_cams=2, verbose=False,
    ) / KP_SCALE
    _scut0 = kp_xyz_fly0[:clip_T, _scut_idx, :]  # post-CSV-override, fly0 = male
    _m = np.isfinite(_male_sam3[:clip_T, 0]) & np.isfinite(_scut0[:, 0])
    if _m.any():
        _d = np.linalg.norm(_male_sam3[:clip_T][_m] - _scut0[_m], axis=1)
        print(f'SAM3 male-COM vs KP-Scutellum  n={int(_m.sum())}  '
              f'median={np.median(_d):.3f} mm  max={np.max(_d):.3f} mm')
except (NameError, FileNotFoundError, OSError, ValueError):
    pass


In [ ]:
# --- Pslow / Pfast classification: method-stages visualization ---
# Walks through the pipeline: detection → alignment → symmetry → PCA → GMM
# → typed centroids. Refits the GMM (random_state=0) so the clusters in
# panels D-F match the labels stored in `pulse_type_results`.
from utils.pulse_classification_stages import make_method_figure

_method_fig = make_method_figure(
    results,
    pulse_type_results,
    fs=song_cfg.fs,
    example_pair_idx=None,   # set to a pair_idx (int) to pick a specific bout
)

_method_pdf = FIG_DIR / "fig_pslow_pfast_method.pdf"
_method_fig.savefig(_method_pdf, dpi=300, bbox_inches="tight")
print(f"saved {_method_pdf}")
plt.show()


In [ ]:
print(np.percentile(ext_pulse_a, [50, 90, 99, 100]))
print(np.percentile(ext_sine_a,  [50, 90, 99, 100]))


# Clip Rendering

In [ ]:
# Add project root to Python path FIRST to ensure our modules take priority
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"Added {project_root} to Python path")
elif sys.path.index(str(project_root)) != 0:
    # Move to front if it exists but isn't first
    sys.path.remove(str(project_root))
    sys.path.insert(0, str(project_root))
    print(f"Moved {project_root} to front of Python path")

import mujoco
from mujoco_visualizer import load_settings, apply_settings, build_scene_option
from mujoco_visualizer.render_settings import apply_scene_flags, get_scene_modifiers
from mujoco_visualizer import load_config
from mujoco_visualizer.presets.fly import FLY_ANATOMY_PATH
import mediapy as media

media.set_show_save_dir(FIG_DIR)    

def make_videos(
    mj_model,
    mj_data,
    qposes_rollout,
    scene_option,
    camera="track1",
    height=512,
    width=512,
    kp_data=None,
    mocap_indices=None,
    kp_anchor_idx=None,
    model_anchor_site_id=None,
    settings=None,
    scene_mods=[],
):
    """
    Make a video of the rollout and reference superimposed.
    
    Args:
        kp_data: Optional (T, N_kp, 3) keypoint positions to overlay as mocap bodies.
        mocap_indices: Optional dict mapping keypoint index to mocap ID (from get_aligned_mocap_indices).
        kp_anchor_idx: Index of anchor keypoint (e.g. Scutellum) in kp_data for alignment.
        model_anchor_site_id: MuJoCo site ID of the corresponding anchor site on the body model.
    """
    frames = []
    with mujoco.Renderer(mj_model, height=height, width=width) as renderer:
        for t in tqdm(range(len(qposes_rollout))):
            mj_data.qpos = qposes_rollout[t]
            mujoco.mj_forward(mj_model, mj_data)
            
            # Update mocap body positions for keypoint overlay
            if kp_data is not None and mocap_indices is not None:
                # Compute offset to align kp anchor to model anchor
                offset = np.zeros(3)
                if kp_anchor_idx is not None and model_anchor_site_id is not None:
                    model_anchor_pos = mj_data.site_xpos[model_anchor_site_id]
                    kp_anchor_pos = kp_data[t, kp_anchor_idx, :]
                    offset = model_anchor_pos - kp_anchor_pos
                
                for kp_idx, mocap_id in mocap_indices.items():
                    mj_data.mocap_pos[mocap_id] = kp_data[t, kp_idx, :] + offset
                    mj_data.mocap_quat[mocap_id] = [1, 0, 0, 0]
            
            renderer.update_scene(
                mj_data, camera=f"{camera}", scene_option=scene_option
            )
            apply_scene_flags(renderer.scene, settings)
            for fn, kw in scene_mods:
                fn(renderer.scene, geom_xpos=mj_data.geom_xpos, **kw)

            pixels = renderer.render()
            frames.append(pixels)
    return frames

In [ ]:
# ── Two-fly rendering: both flies in one scene to show their interaction ─────
# Uses the same visualizer + qpos pipeline as Panel I of the consolidated
# figure (cell 8), so camera presets, per-fly color palettes, floor height,
# and qpos alignment all match. Re-run cell 8 first so `ex`, `data`,
# `VIZ_MODEL_XML`, `VIZ_FLOOR_XML`, `VIZ_SETTINGS_FLY0/1` are defined.
from tqdm.auto import tqdm
from mujoco_visualizer.render_settings import (
    build_scene_option, get_scene_modifiers, apply_scene_flags, load_settings,
)


# ── Pick which bout to render ───────────────────────────────────────────────
# Set to any integer in range(len(results)). Defaults to the exemplar
# index computed in cell 8. Prints pair_idx so you can cross-check.
def find_bout_idx(start_frame, fly_id_prefix=''):
    """Return the results-list index whose info start_frame matches.
    Optionally require ``fly_id_prefix`` to appear in the info fly_id
    (useful when the same start_frame appears in two sessions).
    """
    _sfs = list(info.get('start_frames', []))
    _fids = list(info.get('fly_ids', []))
    for _i, _r in enumerate(results):
        _k = _r.get('tracker_key0') or _r['key0']
        _j = bout_keys.index(_k)
        if int(_sfs[_j]) != int(start_frame):
            continue
        _fid = _fids[_j]
        _fid_s = _fid.decode() if isinstance(_fid, bytes) else str(_fid)
        if fly_id_prefix and fly_id_prefix not in _fid_s:
            continue
        return _i
    raise ValueError(
        f'no results entry with start_frame={start_frame}' +
        (f', fly_id prefix={fly_id_prefix!r}' if fly_id_prefix else ''))

# Example: BOUT_IDX = find_bout_idx(start_frame=803144)
# _TARGET_FLY_ID_PREFIX = 'Session0/2025_10_20_13_20_04'
# _TARGET_START_FRAME   = 446306
# BOUT_IDX = 106
# BOUT_IDX = find_bout_idx(start_frame=803144)
BOUT_IDX = find_bout_idx(start_frame=446306)
_ex = results[BOUT_IDX]
_bout_key = _ex['key0']
_info_j = bout_keys.index(_bout_key)
_fid_raw = list(info.get('fly_ids', []))[_info_j]
_fid_str = (_fid_raw.decode() if isinstance(_fid_raw, bytes)
            else str(_fid_raw))
_start_f = int(list(info.get('start_frames', []))[_info_j])
_end_f   = int(list(info.get('end_frames',   []))[_info_j])
print(f'[render] results[{BOUT_IDX}]  pair_idx={_ex["pair_idx"]}  '
      f'bout={_bout_key}  T={int(_ex["T"])}')
print(f'         fly_id={_fid_str}')
print(f'         frames=[{_start_f}, {_end_f}) in source video')

# ── Clip crop window (set to None to disable) ───────────────────────────────
qpos0_full = np.asarray(data[_ex["key0"]]["qpos"])
qpos1_full = np.asarray(data[_ex["key1"]]["qpos"])
T_full = min(len(qpos0_full), len(qpos1_full))

start_frame_v = 0
end_frame_v   = int((1200 / 1000) * 800)   # None → render to end of bout
start_frame_v = max(0, int(start_frame_v))
end_frame_v   = T_full if end_frame_v is None else min(int(end_frame_v), T_full)
assert end_frame_v > start_frame_v

qpos0 = qpos0_full[start_frame_v:end_frame_v]
qpos1 = qpos1_full[start_frame_v:end_frame_v]
T_render = end_frame_v - start_frame_v
nq = qpos0.shape[1]
print(f'rendering pair_idx={_ex["pair_idx"]}: frames [{start_frame_v}:{end_frame_v}] '
      f'(T={T_render}/{T_full}), nq per fly={nq}')

# ── Build combined model via Panel-I visualizer helper ─────────────────────
# Estimate rig pose from all fly xy across every bout in this session.
# Without this, the Happy_house mesh sits at the MuJoCo world origin while
# the flies live in the DLT calibration frame, and the rig appears offset.
# Restrict to bouts from the CURRENT session — `data` merges multiple
# H5s (Session0 + Session1), each with its own DLT calibration, so
# aggregating xy across sessions mixes disjoint world frames.
# Manual nudge added to the PCA-estimated rig center (cm, world xy).
# Use this to pull the rig toward features like a wall the female climbs.
RIG_CENTER_OFFSET = (0.0, 0.0)

_session_prefix = _fid_str.rsplit('_fly', 1)[0]
_session_keys = [
    bout_keys[_i] for _i, _f in enumerate(info.get('fly_ids', []))
    if _session_prefix in (_f.decode() if isinstance(_f, bytes) else str(_f))
]
print(f'[rig] using {len(_session_keys)}/{len(bout_keys)} bouts from session '
      f'{_session_prefix!r} for pose estimation')
_rig = cfp.estimate_rig_pose_from_qpos(data, bout_keys=_session_keys)
_yaw = _rig['yaw_rad']
_rig_pos  = (_rig['center_xy'][0] + RIG_CENTER_OFFSET[0],
             _rig['center_xy'][1] + RIG_CENTER_OFFSET[1])  # keep XML z
_rig_quat = (np.cos(_yaw / 2), 0.0, 0.0, np.sin(_yaw / 2))
print(f"[rig] center=({_rig['center_xy'][0]:+.3f}, {_rig['center_xy'][1]:+.3f}) cm  "
      f"yaw={np.degrees(_yaw):+.1f}°  "
      f"extent major={_rig['major_len']:.2f} cm  minor={_rig['minor_len']:.2f} cm  "
      f"(rig footprint ≈ 5.00 × 0.58 cm)")
viz = cfp.build_courtship_pair_visualizer(
    flybody_xml=VIZ_MODEL_XML,
    floor_xml=VIZ_FLOOR_XML,
    settings_fly0=VIZ_SETTINGS_FLY0,
    settings_fly1=VIZ_SETTINGS_FLY1,
    rig_pos=_rig_pos,
)
mj_model = viz.model


# Floor-align each fly's root-z so the lowest geom touches the floor per frame.
qpos_pair = np.concatenate([qpos0, qpos1], axis=-1)
# `floor_z_offset` pushes each fly that far below the tight-AABB
# contact point; 1 mm compensates for the sub-pixel claw-tip touch
# that reads as floating at grazing camera angles.
FLOOR_Z_OFFSET = 0.001
qpos_pair = cfp.floor_align_qpos_pair(
    mj_model, qpos_pair, floor_z_offset=FLOOR_Z_OFFSET,
)

# Scene option / modifiers come from the base (scene-wide) preset.
base_settings = load_settings('Earthy_V1_courtship')
scene_option = build_scene_option(base_settings)
scene_option.sitegroup[:] = [1, 1, 1, 0, 0, 0]
scene_mods = get_scene_modifiers(base_settings)

height, width = 448, 1936 // 3



def _stamp_text(frames, text, x=None, y_from_bottom=14,
                font_scale=0.6, thickness=1):
    """Burn ``text`` into each RGB frame (uint8).

    ``x`` is the left-edge pixel column; when ``None`` (default) the text is
    centered horizontally. Negative values are interpreted as offsets from
    the right edge (``w + x``). ``y_from_bottom`` is the baseline offset
    from the image bottom.
    """
    import cv2
    out = np.asarray(frames).copy()
    for i, f in enumerate(out):
        h, w = f.shape[:2]
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX,
                                      font_scale, thickness)
        if x is None:
            xi = (w - tw) // 2
        elif x < 0:
            xi = w + int(x) - tw
        else:
            xi = int(x)
        y = h - y_from_bottom
        cv2.putText(f, text, (xi + 1, y + 1), cv2.FONT_HERSHEY_SIMPLEX,
                    font_scale, (0, 0, 0), thickness + 1, cv2.LINE_AA)
        cv2.putText(f, text, (xi, y), cv2.FONT_HERSHEY_SIMPLEX,
                    font_scale, (255, 255, 255), thickness, cv2.LINE_AA)
        out[i] = f
    return out


def render_two_flies(camera_name):
    _mj_data = mujoco.MjData(mj_model)
    frames = []
    with mujoco.Renderer(mj_model, height=height, width=width) as renderer:
        for t in tqdm(range(T_render), desc=f'cam {camera_name}'):
            _mj_data.qpos[:] = qpos_pair[t]
            mujoco.mj_forward(mj_model, _mj_data)
            renderer.update_scene(_mj_data, camera=camera_name, scene_option=scene_option)
            apply_scene_flags(renderer.scene, base_settings)
            for fn, kw in scene_mods:
                fn(renderer.scene, geom_xpos=_mj_data.geom_xpos, **kw)
            frames.append(renderer.render())
    return frames


# Render from multiple camera angles, side-by-side.
all_frames = []
for cam in np.asarray([0, 2]):
    print(mj_model.camera(cam).name)
    all_frames.append(render_two_flies(mj_model.camera(cam).name))
all_frames = np.concatenate(all_frames, axis=2)
_slow = int(round(fs / 30))
all_frames = _stamp_text(all_frames, f'{_slow}x slowdown')
media.show_video(all_frames, fps=30,
                 title=f'two-fly interaction — pair_idx {_ex["pair_idx"]}')


In [ ]:
# ── Two-fly panning render ──────────────────────────────────────────────────
# Same camera-pan pattern as the two-fly rendering cell above, which must be
# re-run first so viz / mj_model / qpos_pair / T_render / scene_option /
# scene_mods / base_settings are fresh.
from mujoco_visualizer import make_pan_cameras

preset_names    = ['top1', 'side_close3', 'side_close3', 'top4', 'side_close2',]
total_frames    = 960            # one camera per rendered frame
loop_pan        = False          # close the loop back to first preset
segment_weights = None           # [1, 2, 1, 1, 2] relative time per preset (len must match)

pan_cameras = make_pan_cameras(
    mj_model, base_settings,
    preset_names=preset_names,
    total_frames=total_frames,
    segment_weights=segment_weights,
    loop=loop_pan,
)


def make_pan_video_two_flies(
    mj_model, mj_data, qpos_pair, cameras,
    scene_option, scene_mods, settings,
    height=448, width=1936 // 3,
):
    """Two-fly pan render driven by the combined qpos_pair (T, 2*nq)."""
    T = len(cameras)
    assert len(qpos_pair) >= T, f'need >= {T} combined qpos rows, got {len(qpos_pair)}'
    frames = []
    with mujoco.Renderer(mj_model, height=height, width=width) as renderer:
        for t in tqdm(range(T), desc='two-fly pan'):
            mj_data.qpos[:] = qpos_pair[t]
            mujoco.mj_forward(mj_model, mj_data)
            renderer.update_scene(mj_data, camera=cameras[t], scene_option=scene_option)
            apply_scene_flags(renderer.scene, settings)
            for fn, kw in scene_mods:
                fn(renderer.scene, geom_xpos=mj_data.geom_xpos, **kw)
            frames.append(renderer.render())
    return frames


print(f'[pan] rendering results[{BOUT_IDX}]  pair_idx={_ex["pair_idx"]}  '
      f'bout={_bout_key}  fly_id={_fid_str}')
print(f'      source frames=[{_start_f + start_frame_v}, '
      f'{_start_f + end_frame_v})  (bout frames [{start_frame_v}:{end_frame_v}])')
pan_frames_two = make_pan_video_two_flies(
    mj_model, mujoco.MjData(mj_model),
    qpos_pair, pan_cameras,
    scene_option, scene_mods, base_settings,
    height=448, width=1936 // 3,
)
_slow_pan = int(round(fs / 60))
# pan_frames_two = _stamp_text(np.asarray(pan_frames_two), 
#                              f'{_slow_pan}x slowdown',
#                              x=500,)
media.show_video(pan_frames_two, fps=60,
                 title=f'Bout_{_start_f}_{_end_f}_{_slow_pan}xslow')


# Diagonstics: Frame Cropping

In [ ]:
# --- Diagnostic: load strip frames uncropped and overlay projected keypoints ---
# Raw JARVIS 3D output for this session/bout. When set, these override
# kp_xyz_fly0 / kp_xyz_fly1 (both shape (T, 50, 3)) and rebuild center_xyz
# from the raw per-fly Scutellum positions. Leave the paths empty/None to
# fall back to whatever cell 8 produced.
# DATA3D_FLY0_CSV = '/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/Predictions_3D_V3_bout29_test/data3D_fly0.csv'
# DATA3D_FLY1_CSV = '/data2/users/eabe/datasets/Johnson_lab/courtship/Session0/2025_10_20_13_20_04/Predictions_3D_V3_bout29_test/data3D_fly1.csv'
# DATA3D_FLY0_CSV   = SESSION_DIR / 'Predictions_3D_34662592/bout_00009/fly0.csv'
# DATA3D_FLY1_CSV   = SESSION_DIR / 'Predictions_3D_34662592/bout_00009/fly1.csv'
def _load_data3d_csv(path, kp_names_order):
    """Return (xyz, conf) reordered to match ``kp_names_order``.
    The CSV has two header rows (keypoint names repeated 4 times, then
    x/y/z/confidence). Keypoints not in ``kp_names_order`` are dropped;
    keypoints in ``kp_names_order`` but absent from the CSV stay NaN.
    """
    if not path:
        return None, None
    import pandas as _pd
    df = _pd.read_csv(path, header=[0, 1])
    names_top = [c[0] for c in df.columns]
    axes      = [c[1] for c in df.columns]
    arr = df.to_numpy(dtype=float)
    T_rows = arr.shape[0]
    out = np.full((T_rows, len(kp_names_order), 4), np.nan, dtype=float)
    name_to_out = {n: i for i, n in enumerate(kp_names_order)}
    axis_map = {'x': 0, 'y': 1, 'z': 2, 'confidence': 3}
    for ci, (nm, ax) in enumerate(zip(names_top, axes)):
        oi = name_to_out.get(nm)
        if oi is None:
            continue
        aj = axis_map.get(ax.lower())
        if aj is None:
            continue
        out[:, oi, aj] = arr[:, ci]
    return out[..., :3], out[..., 3]

if DATA3D_FLY0_CSV or DATA3D_FLY1_CSV:
    _xyz0, _conf0 = _load_data3d_csv(DATA3D_FLY0_CSV, kp_names)
    _xyz1, _conf1 = _load_data3d_csv(DATA3D_FLY1_CSV, kp_names)
    if _xyz0 is not None:
        kp_xyz_fly0 = _xyz0
        print(f'[csv override] fly0: {_xyz0.shape}  conf mean: {np.nanmean(_conf0):.3f}')
    if _xyz1 is not None:
        kp_xyz_fly1 = _xyz1
        print(f'[csv override] fly1: {_xyz1.shape}  conf mean: {np.nanmean(_conf1):.3f}')
    _scut_idx_dbg = kp_names.index('Scutellum')
    _T_c = min(kp_xyz_fly0.shape[0], kp_xyz_fly1.shape[0], clip_T)
    center_xyz = 0.5 * (
        kp_xyz_fly0[:_T_c, _scut_idx_dbg, :] + kp_xyz_fly1[:_T_c, _scut_idx_dbg, :]
    )
    print(f'[csv override] center_xyz rebuilt: {center_xyz.shape}')

# Renders the N strip frames at full video resolution (no crop), draws every
# fly0/fly1 keypoint projected through DLT, and outlines both the auto-crop
# (yellow dashed) and MANUAL_CROP (green solid). Use this to diagnose whether
# empty-floor tiles are caused by a bad DLT projection, a bad center_xyz, or
# just a bad crop window.
import cv2
import matplotlib.patches as mpatches

MANUAL_CROP = None #(1000,0,600,500)   # set to (x, y, w, h) in raw video pixels to test a crop

# 1. Load strip frames uncropped.
_cap = cfp._open_video(CAM_MP4_PATH)
try:
    _Wv = int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    _Hv = int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames_full = []
    for fidx in frame_idx_strip:
        f = cfp._read_frame(_cap, int(fidx) + int(VIDEO_FRAME_OFFSET), roi=None)
        frames_full.append(f)
finally:
    _cap.release()

# 2. Project every keypoint of both flies to uv for each strip frame.
_scut_idx = kp_names.index("Scutellum")

def _proj_kps(kp_arr, fidxs):
    if kp_arr is None:
        return None
    arr = np.asarray(kp_arr)
    if arr.ndim == 2 and arr.shape[-1] != 3:
        arr = arr.reshape(arr.shape[0], -1, 3)
    out = np.full((len(fidxs), arr.shape[1], 2), np.nan, dtype=float)
    for i, fi in enumerate(fidxs):
        fi = int(fi)
        if 0 <= fi < arr.shape[0]:
            out[i] = cfp._dlt_project(dlt_coeffs, arr[fi] * KP_SCALE)
    return out

uv_fly0 = _proj_kps(kp_xyz_fly0, frame_idx_strip)
uv_fly1 = _proj_kps(kp_xyz_fly1, frame_idx_strip)

# Loud pre-plot diagnostic so a None or NaN state is obvious.
print('=== KP OVERLAY STATE ===')
print(f'  kp_xyz_fly0: {None if kp_xyz_fly0 is None else kp_xyz_fly0.shape}')
print(f'  kp_xyz_fly1: {None if kp_xyz_fly1 is None else kp_xyz_fly1.shape}')
print(f'  uv_fly0:     {None if uv_fly0 is None else uv_fly0.shape}')
print(f'  uv_fly1:     {None if uv_fly1 is None else uv_fly1.shape}')
if uv_fly0 is not None:
    print(f'  uv_fly0[0, Scut]: {uv_fly0[0, _scut_idx]}  '
          f'finite_all: {np.isfinite(uv_fly0).all()}')
if uv_fly1 is not None:
    print(f'  uv_fly1[0, Scut]: {uv_fly1[0, _scut_idx]}  '
          f'finite_all: {np.isfinite(uv_fly1).all()}')
print(f'  KP_SCALE: {KP_SCALE}  dlt_coeffs[:3]: {dlt_coeffs[:3]}')
print('========================')

# 3. Full-frame grid with projection overlay and crop outlines.
N_strip = len(frame_idx_strip)
fig_d, axs_d = plt.subplots(1, N_strip, figsize=(3.0 * N_strip, 3.0))
if N_strip == 1:
    axs_d = [axs_d]
for i, (ax_i, frame, fidx) in enumerate(zip(axs_d, frames_full, frame_idx_strip)):
    if frame is None:
        ax_i.text(0.5, 0.5, f"frame {fidx} N/A", ha='center', va='center',
                  transform=ax_i.transAxes)
        ax_i.axis('off'); continue
    ax_i.imshow(frame)
    if uv_fly0 is not None:
        pts = uv_fly0[i]
        ax_i.scatter(pts[:, 0], pts[:, 1], s=8, c='r', alpha=0.9,
                     edgecolors='none', clip_on=False,
                     label='fly0' if i == 0 else None)
        ax_i.scatter(pts[_scut_idx, 0], pts[_scut_idx, 1], s=80, c='r',
                     marker='*', edgecolors='k', linewidths=0.4, clip_on=False)
    if uv_fly1 is not None:
        pts = uv_fly1[i]
        ax_i.scatter(pts[:, 0], pts[:, 1], s=8, c='b', alpha=0.9,
                     edgecolors='none', clip_on=False,
                     label='fly1' if i == 0 else None)
        ax_i.scatter(pts[_scut_idx, 0], pts[_scut_idx, 1], s=80, c='b',
                     marker='*', edgecolors='k', linewidths=0.4, clip_on=False)

    # Static VIDEO_ROI outline (the crop the panels actually use).
    if 'VIDEO_ROI' in globals() and VIDEO_ROI is not None:
        x, y, w, h = VIDEO_ROI
        ax_i.add_patch(mpatches.Rectangle((x, y), w, h, fill=False,
                                          edgecolor='yellow', ls='--', lw=1.2))
    if MANUAL_CROP is not None:
        x, y, w, h = MANUAL_CROP
        ax_i.add_patch(mpatches.Rectangle((x, y), w, h, fill=False,
                                          edgecolor='lime', ls='-', lw=1.2))

    scut_uv_mid = None
    if uv_fly0 is not None and uv_fly1 is not None:
        scut_uv_mid = 0.5 * (uv_fly0[i, _scut_idx] + uv_fly1[i, _scut_idx])
    elif uv_fly0 is not None:
        scut_uv_mid = uv_fly0[i, _scut_idx]
    elif uv_fly1 is not None:
        scut_uv_mid = uv_fly1[i, _scut_idx]
    t_ms = int(round(int(fidx) / fs * 1000))
    if scut_uv_mid is not None and np.all(np.isfinite(scut_uv_mid)):
        ax_i.set_title(f"f={int(fidx)}  t={t_ms}ms\n"
                       f"uv_mid=({scut_uv_mid[0]:.0f},{scut_uv_mid[1]:.0f})",
                       fontsize=8)
    else:
        ax_i.set_title(f"f={int(fidx)}  uv_mid=NaN", fontsize=8)
    ax_i.set_xticks([]); ax_i.set_yticks([])
    ax_i.autoscale_view()
if uv_fly0 is not None or uv_fly1 is not None:
    axs_d[0].legend(fontsize=7, loc='lower right')
fig_d.suptitle('Full 512x512 frames + DLT-projected keypoints '
               '(yellow=VIDEO_ROI, green=MANUAL_CROP)', fontsize=10)
fig_d.tight_layout(); plt.show()

# 4. Manual-crop preview row (only when MANUAL_CROP is set).
if MANUAL_CROP is not None:
    mx, my, mw, mh = MANUAL_CROP
    fig_m, axs_m = plt.subplots(1, N_strip, figsize=(2.5 * N_strip, 2.5))
    if N_strip == 1:
        axs_m = [axs_m]
    for i, (ax_i, frame, fidx) in enumerate(zip(axs_m, frames_full, frame_idx_strip)):
        if frame is None:
            ax_i.text(0.5, 0.5, f"frame {fidx} N/A", ha='center', va='center',
                      transform=ax_i.transAxes)
            ax_i.axis('off'); continue
        crop = frame[my:my + mh, mx:mx + mw]
        ax_i.imshow(crop)
        if uv_fly0 is not None:
            pts = uv_fly0[i] - np.array([mx, my])
            ax_i.scatter(pts[:, 0], pts[:, 1], s=6, c='r', alpha=0.8,
                         edgecolors='none')
        if uv_fly1 is not None:
            pts = uv_fly1[i] - np.array([mx, my])
            ax_i.scatter(pts[:, 0], pts[:, 1], s=6, c='b', alpha=0.8,
                         edgecolors='none')
        ax_i.set_xlim(0, mw); ax_i.set_ylim(mh, 0)
        ax_i.set_xticks([]); ax_i.set_yticks([])
        ax_i.set_title(f"f={int(fidx)}", fontsize=8)
    fig_m.suptitle(f'MANUAL_CROP = {MANUAL_CROP}', fontsize=10)
    fig_m.tight_layout(); plt.show()

# 5. Diagnostic summary.
print(f'video size: {_Wv} x {_Hv}')
for _tag, _kp in (('fly0', kp_xyz_fly0), ('fly1', kp_xyz_fly1)):
    if _kp is None:
        print(f'  {_tag}: None'); continue
    _a = np.asarray(_kp)
    print(f'  {_tag}: shape {_a.shape}  '
          f'x[{np.nanmin(_a[...,0]):.2f},{np.nanmax(_a[...,0]):.2f}]  '
          f'y[{np.nanmin(_a[...,1]):.2f},{np.nanmax(_a[...,1]):.2f}]  '
          f'z[{np.nanmin(_a[...,2]):.2f},{np.nanmax(_a[...,2]):.2f}]')
print('per-frame Scutellum midpoint:')
for fidx in frame_idx_strip:
    c = center_xyz[int(fidx)]
    uv = cfp._dlt_project(dlt_coeffs, c * KP_SCALE)
    inside = (0 <= uv[0] < _Wv) and (0 <= uv[1] < _Hv)
    print(f'  f={int(fidx):4d}  xyz=({c[0]:7.2f},{c[1]:7.2f},{c[2]:7.2f})  '
          f'uv=({uv[0]:7.1f},{uv[1]:7.1f})  inside={inside}')
# Body-model kp_data baseline (re-centered origin) for contrast.
_kd = np.asarray(data[ex["key0"]]["kp_data"])
if _kd.ndim == 2 and _kd.shape[-1] != 3:
    _kd = _kd.reshape(_kd.shape[0], -1, 3)
print(f'kp_data (fly0, body-model) frame0 Scutellum xyz: {_kd[0, _scut_idx, :]}')

print('uv range across strip (pixels):')
for _tag, _uv in (('fly0', uv_fly0), ('fly1', uv_fly1)):
    if _uv is None:
        print(f'  {_tag}: None'); continue
    _finite = np.isfinite(_uv).all(axis=-1)
    if not _finite.any():
        print(f'  {_tag}: all NaN'); continue
    _f = _uv[_finite]
    print(f'  {_tag}: u[{_f[...,0].min():.1f},{_f[...,0].max():.1f}]  '
          f'v[{_f[...,1].min():.1f},{_f[...,1].max():.1f}]  '
          f'(frame 0 Scutellum uv = {_uv[0, _scut_idx]})')


In [ ]:
frame_idx_strip[: len(axd["video"])]

**Figure N. Pipeline and mechanism summary for a Drosophila courtship bout.** **(A)** Single top-down video frame with body-axis keypoints labeled (Scutellum, WingL_V13, WingR_V13) on the focal male. **(B)** Per-frame wing-V13 z-position (L, purple; R, blue) over the exemplar bout, with scutellum z stacked below. Pulse and sine song segments are shaded behind the traces (salmon = pulse, teal = sine); vertical lines mark individual pulse events colored by sub-type (Pslow = orange, Pfast = deep blue). **(C)** Four video frames sampled across the bout, overlaid with projected keypoints (red = male, blue = female) and the SAM3 segmentation masks at the same alpha-blended colors, demonstrating per-fly detection and pose reconstruction. **(D)** In-phase wing angle (left − right) during the sine-song window used for Panel E. **(E)** Polar histogram of the L − R wing-phase difference across all sine pulses in the bout; concentration near 0 rad confirms bilateral wing-phase coupling during sine song. **(F)** 2-D density of wing-base joint angles during extension vs folding, showing the characteristic sweep direction. **(G)** Pulse-waveform classification: mean ± SD of pooled Pslow and Pfast waveforms fit by a two-component GMM pooled across all pairs. **(H)** Scutellum z-height distribution during singing bouts vs free walking, showing the postural elevation associated with courtship. **(I)** Four MuJoCo-rendered poses (IK fit to the bout's 3D keypoints), floor-aligned, spanning the same time range as Panel C. **(J)** Male thorax pitch (red; derived from the fit quaternion, positive = nose-up) and target pitch (blue; elevation from the male scutellum to the DLT-triangulated female center-of-mass) over the bout. Pulse/sine segments are shaded as in Panel B; where the two traces coincide, the male's body axis is aimed at the female. **(K)** Across all N courtship bouts, violin + per-bout dots of median |body − target pitch| (aim error, degrees); the exemplar bout is highlighted in red.